# Cell Insertion, Desktop Quick-Pickers, Slash-Morph & Image Drop (`CRAFT.js`, `CRAFT.css`)

> Every way a desktop user creates or morphs cells: the active cell-type picker (`window.craftInsertPreview`), hover-edge drag-to-insert and 4-option fan picker, `Shift+A`/`Shift+B` seam quick-picker, `/c` `/n` `/p` `/r` slash-morph and click-margin escape (`window.craftDesktopUX`), and native Monaco image-drop handoff (`window.craftImageDrop`).

## Insertion Templates & Cell-Type Config (`#| export_ft`)

Single source of truth for cell-type SVG icons, insertion triggers, fan picker markup, and the `key_to_type` / `slash_map` tables exposed in `window.__craftCfg`.

In [ ]:
#| export_ft CRAFT.js
#| eval: false
CELL_PATHS = dict(
    Code='M8 6l-6 6 6 6m8-12 6 6-6 6M14 4l-4 16',
    Note='M5 3h10l4 4v14H5z M15 3v5h4M8 12h8m-8 4h8',
    Raw='M3 5h18v14H3z M7 9v6',
    Prompt='M3 4h18v12H9l-6 4z M7 10h.01m5 0h.01m5 0h.01',
)
DESK_ORDER = ('Code', 'Note', 'Prompt', 'Raw')

TEMPLATES = merge(
    dict(
        insert_trigger=Button(id='craft-insert-preview', klass='craft-nav-icon-btn', type='button', hidden=True, popovertarget='craft-insert-preview-menu'),
        insert_menu=craft_menu(
            Small('New cell type'),
            *[Button(svg_icon(p, 15, 1.6), name, type='button', data_kind=name) for name, p in CELL_PATHS.items()],
            id='craft-insert-preview-menu', popover='auto'),
        desk_edge_below=Button(
            Span(svg_icon('M12 6v12M6 12h12', 12, 1.8), Span(klass='craft-desk-edge-kind'), klass='craft-desk-edge-pill'),
            type='button', klass='craft-desk-edge', data_edge='below',
            aria_label='Insert cell below (drag to pick type)'),
        desk_picker=Div(
            Div(klass='craft-type-picker-head craft-desk-picker-head'),
            Div(*[craft_btn(svg_icon(CELL_PATHS[t], 14, 1.8), Span(t), ft('kbd', t[0]),
                            role='menuitem', data_type=t) for t in DESK_ORDER],
                klass='craft-type-picker-opts craft-desk-picker-opts'),
            id='craft-desk-picker', klass='craft-type-picker', role='menu'),
        insert_placeholder=Div(klass='craft-insert-placeholder'),
    ),
    {f'cell_icon_{k}': svg_icon(p, 15, 1.6) for k, p in CELL_PATHS.items()},
    {f'desk_icon_{k}': svg_icon(p, 11, 1.8) for k, p in CELL_PATHS.items()},
)

CONFIG = dict(
    cell_types=list(DESK_ORDER),
    key_to_type={t[0].lower(): t for t in DESK_ORDER},
    slash_map={k: t for t in DESK_ORDER for k in (t[0].lower(), t.lower())},
)

## Insert Type Picker (`window.craftInsertPreview`)

Controls which cell type `a` and `b` insert, renders the shimmer placeholder while waiting for Solveit's WebSocket card refresh, and hands focus to `craftC2` or `craftMd` once the new card arrives.

In [ ]:
#| export_js CRAFT.js
#| eval: false
(() => {
  window.craftInsertPreview?.disable();
  if (!document.getElementById('dialog-container')) return;
  const types = {Code: 1, Note: 1, Raw: 1, Prompt: 1};
  const icon = name => window.__craftTpl?.[`cell_icon_${name}`] || '';
  const key = 'craft-insert-preview:v1:' + (new URLSearchParams(location.search).get('name') || location.pathname);
  const {mountTpl, qs, qsa, on, store, patch, onDomChange} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const trigger = mountTpl('insert_trigger'), menu = mountTpl('insert_menu');
  let current;
  const select = name => {
    current = name;
    const title = `A/B inserts: ${name}`;
    patch(trigger, {innerHTML: icon(name), title, 'aria-label': title});
    for (const option of qsa('button', menu)) option.setAttribute('aria-pressed', String(option.dataset.kind === name));
  };
  const saved = store.get(key);
  select(Object.hasOwn(types, saved) ? saved : document.getElementById('default_code')?.checked ? 'Code' : 'Prompt');
  let scroller;
  const position = () => {
    if (!scroller) return;
    const rect = scroller.getBoundingClientRect();
    trigger.hidden = !rect.width || !rect.height;
    Object.assign(trigger.style, {left: 'auto', right: '42px', top: '6px'});
    Object.assign(menu.style, {right: '10px', top: '36px'});
    window.craftMobile?.reposition?.();
  };
  const resize = new ResizeObserver(position);
  const sync = () => {
    const next = document.getElementById('dialog-container');
    if (scroller === next) return;
    if (scroller) {
      resize.unobserve(scroller);
      scroller.classList.remove('craft-insert-preview');
    }
    scroller = next;
    if (!scroller) { trigger.hidden = true; return; }
    scroller.classList.add('craft-insert-preview');
    resize.observe(scroller); position();
  };
  const choose = name => {
    select(name);
    store.set(key, name);
  };
  const activeCells = () => qsa('.editable:is([data-sm="primary"],[data-sm="secondary"])', scroller).filter(el => el.getClientRects().length);
  const cellType = el => el?.querySelector('.card-in[data-mtype]')?.dataset.mtype;
  const editorCell = () => (document.querySelector('#full_editor #id_')?.value || '').replace(/^_/, '');
  const notify = error => {
    console.error('CRAFT insertion:', error);
    window.__craftCMCore.notify(error.message, {status: 'danger'});
  };
  const waitFor = (check, timeout = 5000) => {
    if (ac.signal.aborted) return Promise.resolve(null);
    const immediate = check();
    if (immediate) return Promise.resolve(immediate);
    return new Promise(resolve => {
      let done = false;
      const onAbort = () => finish(null);
      const finish = val => {
        if (done) return;
        done = true;
        ac.signal.removeEventListener('abort', onAbort);
        offDomWait();
        clearInterval(tick);
        clearTimeout(cap);
        resolve(val);
      };
      const probe = () => { const r = check(); if (r) finish(r); };
      ac.signal.addEventListener('abort', onAbort, {once: true});
      const offDomWait = onDomChange(probe, {attrs: ['data-mtype', 'class', 'hidden', 'data-editing'], delay: 0});
      const tick = setInterval(probe, 100);
      const cap = setTimeout(() => finish(null), timeout);
    });
  };
  const setType = async (id, type) => {
    if (cellType(document.getElementById(id)) !== type) {
      await window.__craftCMCore.htmxPost('/set_mtyp_', {id_: id, ids: id, mtyp: type});
      if (!(await waitFor(() => cellType(document.getElementById(id)) === type))) throw new Error(`Could not switch cell to ${type}.`);
    }
  };
  const blocked = () => window.__craftCMCore.typingBlocked('.cm-editor,#craft-insert-preview-menu');
  let busy = false;
  const execute = action => {
    if (!busy) {
      busy = true;
      Promise.resolve().then(action).catch(notify).finally(() => { busy = false; });
    }
    return true;
  };
  const mode = name => {
    if (blocked()) return false;
    choose(name);
    const cells = activeCells();
    if (cells.length !== 1) return true;
    return execute(async () => { await setType(cells[0].id, name.toLowerCase()); });
  };
  // opts.type ('Code'|'Note'|'Prompt'|'Raw') overrides the picker; opts.cell targets a cell other than the
  // selection; opts.force skips the focus guard (the mobile toolbar has already closed any editor).
  const insert = (where, opts = {}) => {
    if (!opts.force && blocked()) return false;
    const primary = opts.cell || activeCells().find(el => el.dataset.sm === 'primary');
    const button = primary?.querySelector(`[hx-post="/add_${where}_"]`);
    if (!button) return true;
    const name = Object.hasOwn(types, opts.type || '') ? opts.type : current;
    return execute(async () => {
      const type = name.toLowerCase(), old = new Set([...scroller.querySelectorAll('.editable[id]')].map(el => el.id));
      if (!window.craftC2?.guardNonCodeInsertion) throw new Error('Reload this dialog to enable typed insertion.');
      if (type === 'code') {
        if (!window.craftC2?.expectCodeInsertion) throw new Error('Reload this dialog to enable C2 insertion.');
        window.craftC2.expectCodeInsertion();
      }
      const release = type !== 'code' ? window.craftC2.guardNonCodeInsertion(old) : null;
      const unhide = window.craftIdleComposer?.holdForInsertion();
      const ph = window.__craftCMCore.mountTpl('insert_placeholder');
      if (where === 'above') primary.before(ph); else primary.after(ph);
      let id = null;
      try {
        button.dispatchEvent(new MouseEvent('click', {bubbles:false}));
        id = await waitFor(() => {
          const editing = editorCell(), id = '_' + editing;
          if (editing && !old.has(id) && document.getElementById(id)) return id;
          if (type === 'code') return [...scroller.querySelectorAll('.editable[id]')].find(el => !old.has(el.id) && window.craftC2?.manages(el.id))?.id;
        });
        if (!id) throw new Error(`Could not confirm ${name} insertion.`);
        if (type === 'code') {
          if (!(await waitFor(() => cellType(document.getElementById(id)) === 'code' && editorCell() !== id.slice(1) && Boolean(document.getElementById(id)?.querySelector('.craft-c2-host'))))) throw new Error('Could not confirm C2 handoff.');
        } else {
          await setType(id, type);
          const cancel = await waitFor(() => {
            if (editorCell() !== id.slice(1) || window.edValue?.()) return true;
            const btn = document.getElementById('cancel_btn');
            return btn && !btn.disabled ? btn : null;
          });
          if (!cancel) throw new Error('Could not close the insertion composer.');
          if (cancel && cancel !== true) {
            const field = document.querySelector('#full_editor #msg_type');
            if (field) field.value = type;
            cancel.click();
            if (!(await waitFor(() => editorCell() !== id.slice(1)))) throw new Error('Could not close the insertion composer.');
            if (cellType(document.getElementById(id)) !== type) throw new Error(`New cell did not retain type ${type}.`);
          }
        }
      } finally { ph.remove(); release?.(); unhide?.(); }
      if (id) {
        document.getElementById(id)?.scrollIntoView({block: 'nearest'});
        if (type === 'code') {
          if (!document.activeElement?.closest?.(`#${id} .craft-c2-host`)) window.craftC2?.edit?.(id, {anchor: 0, head: 0});
        } else if (!document.activeElement?.closest?.(`#${id} .craft-md-host`)) {
          window.craftMd?.edit?.(id, {toEnd: true});
        }
      }
    });
  };
  const defs = [
    ['si-a', 'a', 'Add message above', () => insert('above')],
    ['si-b', 'b', 'Add message below', () => insert('below')],
    ...['Code','Note','Prompt','Raw'].map((name,i) => [`si-mode-${name.toLowerCase()}`, `Mod+Shift+${'JKL;'[i]}`, `Switch message to ${name.toLowerCase()}`, () => mode(name)]),
  ].map(([id,combo,label,handler]) => ({id,combo,label,handler,group:'Solveit · Dialog (selection mode)'}));
  const registerBindings = () => defs.forEach(spec => window.__hotkeyRegister?.(spec));
  const builtins = qs('script[data-solveit-hotkeys="builtins"]');
  on(builtins, 'load', registerBindings, {once: true, signal: ac.signal});
  registerBindings();
  on(menu, 'click', e => {
    const option = e.target.closest('button[data-kind]');
    if (!option) return;
    choose(option.dataset.kind);
    menu.hidePopover(); trigger.focus();
  }, sig);
  document.body.append(trigger, menu);
  const KIND_MAP = {code: 'Code', note: 'Note', prompt: 'Prompt', raw: 'Raw'};
  const onHtmxConfig = e => {
    const path = e.detail?.path || '';
    const mtyp = e.detail?.parameters?.mtyp || e.detail?.parameters?.msg_type;
    if ((path.includes('set_mtyp') || path.includes('upsert_msg')) && mtyp && KIND_MAP[String(mtyp).toLowerCase()]) {
      choose(KIND_MAP[String(mtyp).toLowerCase()]);
    }
  };
  const onTabClick = e => {
    const tab = e.target?.closest?.('#full_editor [role="tab"], #full_editor button');
    const name = tab && KIND_MAP[(tab.textContent || '').trim().toLowerCase()];
    if (name) choose(name);
  };
  on(document, 'htmx:configRequest', onHtmxConfig, sig);
  on(document, 'click', onTabClick, cap);
  const offDom = onDomChange(sync, {delay: 0});
  on(window, 'resize', position, sig);
  window.craftInsertPreview = {
    insert: (where, opts) => insert(where, opts),
    choose,
    setType,
    type: () => current,
    isBusy: () => busy,
    disable() {
    ac.abort();
    offDom(); resize.disconnect();
    defs.forEach(spec => window.__hotkeyRegister?.({...spec, builtin:true, handler:() => true}));
    scroller?.classList.remove('craft-insert-preview');
    trigger.remove(); menu.remove(); delete window.craftInsertPreview;
  }};
  sync();
})();

## Desktop UX (`window.craftDesktopUX`)

Hover-edge `+` handle with drag-to-pick fan menu, `Shift+A`/`Shift+B` seam quick-picker, click-margin/header escape out of active inline editors, and `/c` `/n` `/p` `/r` slash-morph on empty cells.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Desktop UX: hover-edge drag-to-insert, click-margin escape, Shift+A/B seam picker & slash-morph ── */
(() => {
  window.craftDesktopUX?.disable?.();
  if (!document.getElementById('dialog-container')) return;
  const {mountTpl, on} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const isMobile = () => !!window.craftMobile?.active?.();
  const dialog = () => document.getElementById('dialog-container');
  const CELL_TYPES = window.__craftCfg?.cell_types || ['Code', 'Note', 'Prompt', 'Raw'];
  const KEY_TO_TYPE = window.__craftCfg?.key_to_type || {c: 'Code', n: 'Note', p: 'Prompt', r: 'Raw'};
  const SLASH_RE = /^\/(c|code|n|note|p|prompt|r|raw)[ \t\n]$/i;
  const SLASH_MAP = window.__craftCfg?.slash_map || {c: 'Code', code: 'Code', n: 'Note', note: 'Note', p: 'Prompt', prompt: 'Prompt', r: 'Raw', raw: 'Raw'};
  /* Bottom hover-edge + handle on desktop (shown when hovering near bottom-center) */
  const hBelow = mountTpl('desk_edge_below');
  const syncHandleKinds = () => {
    const t = window.craftInsertPreview?.type?.() || 'Code';
    const k = hBelow.querySelector('.craft-desk-edge-kind');
    if (k) k.innerHTML = window.__craftTpl?.[`desk_icon_${t}`] || window.__craftTpl?.desk_icon_Code || '';
    hBelow.title = `Click to insert ${t} below · Drag to choose type`;
  };

  /* Shared 4-option fan picker for both drag-to-insert and Shift+A/B */
  const picker = mountTpl('desk_picker');
  document.body.append(picker);

  let hostCell = null, pState = null; // {edge, cell, pointerId, startX, startY, dragged, hotType, keyMode}
  const detachHandles = () => {
    hBelow.removeAttribute('data-near');
    hostCell?.classList.remove('craft-desk-host');
    hBelow.remove();
    hostCell = null;
  };
  const attachHandles = cell => {
    if (isMobile() || !cell?.isConnected) { detachHandles(); return; }
    if (cell === hostCell) { syncHandleKinds(); return; }
    detachHandles();
    hostCell = cell;
    cell.classList.add('craft-desk-host');
    syncHandleKinds();
    cell.append(hBelow);
  };
  const setHot = type => {
    if (pState) pState.hotType = type;
    for (const b of picker.querySelectorAll('button[data-type]')) {
      b.toggleAttribute('data-hot', b.dataset.type === type);
    }
  };
  const closePicker = () => {
    hBelow.removeAttribute('data-active');
    picker.classList.remove('open');
    setHot(null);
    pState = null;
  };
  const positionPicker = (anchorRect, edge, hint) => {
    picker.querySelector('.craft-desk-picker-head').textContent = hint || (edge === 'above' ? 'Insert above' : 'Insert below');
    picker.classList.add('open');
    const pw = picker.offsetWidth || 270, ph = picker.offsetHeight || 68;
    const cx = anchorRect.left + anchorRect.width / 2;
    const left = Math.max(8, Math.min(cx - pw / 2, innerWidth - pw - 8));
    let top;
    if (edge === 'above') {
      const aboveTop = anchorRect.top - ph - 10;
      top = aboveTop >= 8 ? aboveTop : Math.min(anchorRect.bottom + 10, innerHeight - ph - 8);
    } else {
      const belowTop = anchorRect.bottom + 10;
      top = (belowTop + ph <= innerHeight - 8) ? belowTop : Math.max(8, anchorRect.top - ph - 10);
    }
    picker.style.left = Math.round(left) + 'px';
    picker.style.top = Math.round(top) + 'px';
  };
  const hitOpt = (x, y) => {
    const PAD = 8;
    for (const b of picker.querySelectorAll('button[data-type]')) {
      const r = b.getBoundingClientRect();
      if (x >= r.left - PAD && x <= r.right + PAD && y >= r.top - PAD && y <= r.bottom + PAD) return b.dataset.type;
    }
    return null;
  };
  async function commitEditors() {
    if (window.craftC2?.active?.()) await window.craftC2.exit();
    if (window.craftMd?.active?.()) await window.craftMd.exit();
  }
  async function doInsert(edge, cell, type) {
    if (type) window.craftInsertPreview?.choose?.(type);
    syncHandleKinds();
    await commitEditors();
    window.craftInsertPreview?.insert?.(edge, {type: type || window.craftInsertPreview?.type?.(), cell, force: true});
  }

  const onHandleDown = e => {
    if (isMobile() || e.button !== 0) return;
    const handle = e.currentTarget;
    const cell = handle.closest('#dialog-container > .editable[id]');
    if (!cell) return;
    e.preventDefault(); e.stopPropagation();
    closePicker();
    const edge = handle.dataset.edge;
    handle.dataset.active = 'true';
    handle.setPointerCapture?.(e.pointerId);
    pState = {edge, cell, pointerId: e.pointerId, startX: e.clientX, startY: e.clientY, dragged: false, hotType: null, keyMode: false};
  };
  const onHandleMove = e => {
    if (!pState || e.pointerId !== pState.pointerId || pState.keyMode) return;
    const dist = Math.hypot(e.clientX - pState.startX, e.clientY - pState.startY);
    if (!pState.dragged && dist > 6) {
      pState.dragged = true;
      positionPicker(e.currentTarget.getBoundingClientRect(), pState.edge, `Drop to insert ${pState.edge} & set default`);
    }
    if (pState.dragged) setHot(hitOpt(e.clientX, e.clientY));
  };
  const onHandleUp = e => {
    if (!pState || e.pointerId !== pState.pointerId || pState.keyMode) return;
    const {edge, cell, dragged, hotType} = pState;
    closePicker();
    if (!dragged) doInsert(edge, cell, null);
    else if (hotType) doInsert(edge, cell, hotType);
  };
  const onHandleCancel = e => { if (pState && e.pointerId === pState.pointerId) closePicker(); };
  const onHandleClick = e => { e.preventDefault(); e.stopPropagation(); };
  on(hBelow, {pointerdown: onHandleDown, pointermove: onHandleMove, pointerup: onHandleUp, pointercancel: onHandleCancel, click: onHandleClick}, sig);
  const onPickerClick = e => {
    const b = e.target.closest('button[data-type]');
    if (!b || !pState) return;
    e.preventDefault(); e.stopPropagation();
    const {edge, cell} = pState;
    closePicker();
    doInsert(edge, cell, b.dataset.type);
  };
  on(picker, 'click', onPickerClick, sig);

  /* Track cursor proximity to the bottom-center target area of a cell on desktop */
  let moveRaf = 0, lastMoveEv = null;
  const flushMouseMove = () => {
    moveRaf = 0;
    const e = lastMoveEv;
    lastMoveEv = null;
    if (!e || isMobile() || pState) return;
    const t = e.target;
    if (t instanceof Element && hBelow.contains(t)) {
      hBelow.dataset.near = 'true';
      return;
    }
    const cell = t instanceof Element ? t.closest('#dialog-container > .editable[id]') : null;
    if (cell) {
      attachHandles(cell);
      if (hostCell === cell) {
        const r = cell.getBoundingClientRect();
        const nearY = Math.abs(e.clientY - r.bottom) <= 26;
        const nearX = Math.abs(e.clientX - (r.left + r.width / 2)) <= 140;
        hBelow.toggleAttribute('data-near', nearY && nearX);
      }
    } else if (hostCell) {
      const r = hostCell.getBoundingClientRect();
      const nearY = Math.abs(e.clientY - r.bottom) <= 26;
      const nearX = Math.abs(e.clientX - (r.left + r.width / 2)) <= 140;
      if (nearY && nearX) hBelow.dataset.near = 'true';
      else hBelow.removeAttribute('data-near');
    }
  };
  const onMouseMove = e => {
    if (isMobile() || pState) return;
    lastMoveEv = e;
    if (!moveRaf) moveRaf = requestAnimationFrame(flushMouseMove);
  };
  const onDialogLeave = () => { if (!pState) detachHandles(); };
  on(document, 'mousemove', onMouseMove, {passive: true, signal: ac.signal});
  on(dialog(), 'mouseleave', onDialogLeave, sig);

  /* Click-margin / click-header to escape active inline editors on desktop */
  const onDocMouseDown = e => {
    if (isMobile()) return;
    if (pState?.keyMode && !picker.contains(e.target)) closePicker();
    if (e.button !== 0) return;
    const hasEd = !!(window.craftC2?.active?.() || window.craftMd?.active?.());
    if (!hasEd) return;
    const t = e.target;
    if (!(t instanceof Element)) return;
    if (t.closest('#solveit-file-tree-panel, #solveit-temp-editor, #solveit-launcher, .craft-action-menu, #craft-desk-picker, .craft-desk-edge, .cm-editor, .cm-tooltip, .uk-modal, .uk-dropdown, .uk-drop, nav.monster-navbar')) return;
    const inDialog = t.closest('#dialog-container');
    if (!inDialog) return;
    const inCell = t.closest('#dialog-container > .editable[id]');
    const onHeaderBlank = inCell && t.closest('.uk-card-header') && !t.closest('button, a, input, select');
    if (!inCell || onHeaderBlank) {
      commitEditors();
    }
  };
  on(document, 'mousedown', onDocMouseDown, cap);

  /* Shift+A / Shift+B inline seam quick-picker in selection mode */
  const blocked = () => window.__craftCMCore.typingBlocked('.cm-editor');
  const onKeyDown = e => {
    if (isMobile() || e.isComposing) return;
    if (pState?.keyMode) {
      if (e.key === 'Escape') { e.preventDefault(); e.stopImmediatePropagation(); closePicker(); return; }
      const t = KEY_TO_TYPE[e.key.toLowerCase()];
      if (t && !e.metaKey && !e.ctrlKey && !e.altKey) {
        e.preventDefault(); e.stopImmediatePropagation();
        const {edge, cell} = pState;
        closePicker();
        doInsert(edge, cell, t);
        return;
      }
      return;
    }
    if (e.shiftKey && !e.metaKey && !e.ctrlKey && !e.altKey && (e.key === 'A' || e.key === 'B')) {
      if (blocked()) return;
      const cell = dialog()?.querySelector(':scope > .editable[data-sm="primary"]');
      if (!cell) return;
      e.preventDefault(); e.stopImmediatePropagation();
      const edge = e.key === 'A' ? 'above' : 'below';
      const r = cell.getBoundingClientRect();
      const seamRect = edge === 'above'
        ? {left: r.left, width: r.width, top: r.top, bottom: r.top}
        : {left: r.left, width: r.width, top: r.bottom, bottom: r.bottom};
      pState = {edge, cell, pointerId: null, startX: 0, startY: 0, dragged: false, hotType: null, keyMode: true};
      positionPicker(seamRect, edge, `Insert ${edge} · press C / N / P / R`);
    }
  };
  on(document, 'keydown', onKeyDown, cap);

  /* Slash-morph on empty cells: typing /c, /code, /n, /note, /p, /prompt, /r, /raw + Space/Enter morphs the cell */
  let morphing = false;
  const checkSlashMorph = () => {
    if (morphing || isMobile()) return;
    let id = null, view = null;
    const c2Id = window.craftC2?.active?.();
    if (c2Id) {
      view = window.craftC2.view(c2Id);
      if (view?.hasFocus) id = '_' + String(c2Id).replace(/^_/, '');
    }
    if (!id) {
      const mdKey = window.craftMd?.active?.();
      if (mdKey && mdKey.endsWith(':in')) {
        view = window.craftMd.view(mdKey);
        if (view?.hasFocus) id = '_' + mdKey.split(':')[0].replace(/^_/, '');
      }
    }
    if (!id || !view) return;
    const text = view.state.doc.toString();
    const m = SLASH_RE.exec(text);
    if (!m) return;
    const targetName = SLASH_MAP[m[1].toLowerCase()];
    const targetType = targetName.toLowerCase();
    const el = document.getElementById(id);
    const curType = el?.querySelector('.card-in[data-mtype]')?.dataset.mtype;
    if (!el || curType === targetType) return;
    morphing = true;
    view.dispatch({changes: {from: 0, to: text.length, insert: ''}});
    (async () => {
      try {
        await commitEditors();
        window.craftInsertPreview?.choose?.(targetName);
        if (window.craftInsertPreview?.setType) {
          await window.craftInsertPreview.setType(id, targetType);
        }
        if (targetType === 'code') window.craftC2?.edit?.(id.slice(1), {anchor: 0, head: 0});
        else window.craftMd?.edit?.(id.slice(1), {toEnd: true});
      } catch (err) {
        console.error('craftDesktopUX: slash-morph failed', err);
      } finally { morphing = false; }
    })();
  };
  let slashPend = false;
  const onSlashInput = e => {
    const t = e.target;
    if (!(t instanceof Element) || !t.closest('.cm-content, .cm-editor')) return;
    if (slashPend) return;
    slashPend = true;
    queueMicrotask(() => { slashPend = false; checkSlashMorph(); });
  };
  on(document, 'input', onSlashInput, cap);

  const offMobile = window.craftMobile?.onChange?.(active => { if (active) { closePicker(); detachHandles(); } });
  window.craftDesktopUX = {
    disable() {
      offMobile?.(); closePicker(); detachHandles(); picker.remove(); if (moveRaf) { cancelAnimationFrame(moveRaf); moveRaf = 0; }
      ac.abort();
      delete window.craftDesktopUX;
    },
  };
})();

## Image Drop Handoff (`window.craftImageDrop`)

Routes image files dropped onto a code, note, or prompt card into Solveit's native Monaco attachment flow.

In [ ]:
#| export_js CRAFT.js
#| eval: false
(() => {
  window.craftImageDrop?.disable();
  const {qs, on, notify: coreNotify} = window.__craftCMCore;
  const target = e => {
    if (e.target.closest?.('.monaco-editor')) return;
    return e.target.closest?.('#dialog-container > .editable > .card-in:is([data-mtype="code"],[data-mtype="note"],[data-mtype="prompt"])');
  };
  const field = id => qs(`#full_editor #${id}`)?.value;
  const editorId = () => (field('id_') || '').replace(/^_/, '');
  const model = () => window.editor?.getModel();
  const notify = message => coreNotify(message, {status: 'warning'});
  let busy = false, disabled = false;
  const dragover = e => {
    if (!target(e)) return;
    const items = [...(e.dataTransfer?.items || [])];
    if (!items.some(i => i.kind === 'file' && (!i.type || i.type.startsWith('image/'))) && (items.length || !e.dataTransfer?.types.includes('Files'))) return;
    e.preventDefault();
    e.dataTransfer.dropEffect = 'copy';
  };
  const attach = async (e, card, file) => {
    if (typeof window.handleBlob !== 'function') return;
    e.preventDefault();
    e.stopPropagation();
    if (busy) return notify('Wait for the current image upload to finish.');
    const cell = card.parentElement, id = cell.id.slice(1), kind = card.dataset.mtype;
    const priorModel = model();
    const ready = () => editorId() === id && field('is_input') === '1' && field('msg_type') === kind && model() && !model().uri.path.endsWith('/temp');
    busy = true;
    try {
      if (!ready()) {
        if (editorId() || window.edValue?.()) return notify('Finish editing the open cell before attaching an image to another cell.');
        if (window.craftC2?.active?.() === id) await window.craftC2.exit(id);
        if (window.craftMd?.active?.() === `${id}:in`) await window.craftMd.exit(id, 'in');
        const btn = card.querySelector('[hx-post="/editor_"]');
        if (btn) { btn.dataset.craftAllowMonaco = '1'; btn.click(); delete btn.dataset.craftAllowMonaco; }
        for (let i = 0; i < 100 && !disabled && !(ready() && model() !== priorModel); i++) await new Promise(resolve => setTimeout(resolve, 50));
        if (disabled) return;
        if (!ready() || model() === priorModel) return notify('Could not open the cell to attach the image.');
      }
      window.craftIdleComposer?.reveal();
      window.editor.focus();
      const ext = file.name.match(/\.([^.]+)$/)?.[1] || file.type.split('/')[1] || 'png';
      await window.handleBlob(file, ext, true, file.name);
    } catch (err) {
      console.error('CRAFT image attachment:', err);
      notify('Could not attach the image.');
    } finally { busy = false; }
  };
  const drop = e => {
    const card = target(e), file = e.dataTransfer?.files[0];
    if (card && file?.type.startsWith('image/')) attach(e, card, file);
  };
  const ac = new AbortController();
  on(window, {dragover, drop}, {capture: true, signal: ac.signal});
  window.craftImageDrop = {disable() {
    disabled = true;
    ac.abort();
    delete window.craftImageDrop;
  }};
})();

## Insertion & Fan Picker CSS

In [ ]:
#| export_css CRAFT.css
#| eval: false
html:not(.craft-mobile) #dialog-container > .editable.craft-desk-host {
  position: relative;
  overflow: visible !important; content-visibility: visible !important; contain: none !important;
}
html:not(.craft-mobile) #dialog-container > .editable.craft-desk-host:has(> .craft-desk-edge:is([data-near], :hover, :focus-visible, [data-active="true"]))::after {
  mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000);
  mask-position: 0 0, center bottom;
  mask-size: 100% 100%, 48px 6px;
  mask-repeat: no-repeat;
  mask-composite: exclude;
  -webkit-mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000);
  -webkit-mask-position: 0 0, center bottom;
  -webkit-mask-size: 100% 100%, 48px 6px;
  -webkit-mask-repeat: no-repeat;
  -webkit-mask-composite: xor;
}
.craft-desk-edge { display: none; }
html:not(.craft-mobile) .craft-desk-edge {
  display: flex; align-items: center; justify-content: center;
  position: absolute; left: 50%; z-index: 110;
  width: 64px; height: 22px; padding: 0; border: 0; background: transparent;
  transform: translateX(-50%);
  opacity: 0; pointer-events: none; transition: opacity .12s ease;
  user-select: none; cursor: grab;
}
html:not(.craft-mobile) .craft-desk-edge[data-edge="below"] { bottom: -11px; }
html:not(.craft-mobile) .craft-desk-edge:is([data-near], :hover, :focus-visible, [data-active="true"]) { opacity: 1; pointer-events: auto; }
html:not(.craft-mobile) .craft-desk-edge-pill {
  display: inline-flex; align-items: center; gap: 3px;
  height: 18px; padding: 0 7px; border-radius: 999px;
  background: var(--craft-surface); border: 1px solid var(--craft-accent-border); color: var(--craft-accent);
  box-shadow: 0 2px 8px #0008;
  transition: transform .1s ease, background .1s ease, border-color .1s ease, color .1s ease;
}
html:not(.craft-mobile) .craft-desk-edge-kind { display: inline-flex; align-items: center; opacity: .75; }
html:not(.craft-mobile) .craft-desk-edge:hover .craft-desk-edge-pill,
html:not(.craft-mobile) .craft-desk-edge[data-active="true"] .craft-desk-edge-pill {
  background: var(--craft-border-soft); border-color: var(--craft-accent); color: #c0caf5; transform: scale(1.06);
}

/* Shared desktop (#craft-desk-picker) & mobile (#craft-edge-picker) blurred type-picker shell */
.craft-type-picker {
  display: none; position: fixed; z-index: 1250;
  padding: 6px 8px 8px;
  background: #16161ef2; backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px);
  border: 1px solid #7aa2f755; border-radius: 10px;
  box-shadow: 0 12px 32px #000c; user-select: none; -webkit-user-select: none;
}
.craft-type-picker.open { display: block; }
.craft-type-picker-head {
  font: 500 10px/1 var(--craft-prose-family); color: var(--craft-muted);
  text-align: center; margin-bottom: 6px; letter-spacing: .03em; text-transform: uppercase;
}
.craft-type-picker-opts { display: flex; gap: 5px; }
.craft-type-picker button {
  gap: 5px; height: auto; padding: 6px 9px; border-radius: 7px;
  font: 500 11.5px/1 var(--craft-prose-family); -webkit-tap-highlight-color: transparent;
  transition: transform .08s ease, background .08s ease, border-color .08s ease, color .08s ease;
}
.craft-type-picker button svg { color: var(--craft-accent); }
#craft-desk-picker button kbd {
  font: 600 9.5px/1 var(--craft-code-family); color: var(--craft-muted); opacity: .75;
  padding: 1px 4px; border-radius: 3px; background: var(--craft-recessed);
}
.craft-type-picker button:is(:hover, [data-hot], :active) {
  background: #7aa2f733; border-color: var(--craft-accent); color: var(--craft-fg-strong); transform: translateY(-1px);
}

#dialog-container.craft-insert-preview { box-sizing: border-box; margin-right: 0 !important; min-width: 0; }
#craft-insert-preview[hidden] { display: none; }
.craft-insert-placeholder { height: 3px; margin: 4px 0; border-radius: 999px; background: linear-gradient(90deg, #7aa2f722, var(--craft-accent), var(--craft-cyan), #7aa2f722); background-size: 200% 100%; animation: craft-ins-shimmer 1s linear infinite; pointer-events: none; }
@keyframes craft-ins-shimmer { from { background-position: 200% 0; } to { background-position: -200% 0; } }

#craft-insert-preview-menu small { display: block; padding: 4px 10px; color: #9b9cab; }

Verify `09_cell_insertion.ipynb` exports insertion modules, config, and CSS.

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/09_cell_insertion.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
for sym in ('window.craftInsertPreview', 'window.craftDesktopUX', 'window.craftImageDrop', '__craftCfg'):
    assert sym in js, f"Missing {sym}"
css = '\n'.join(code for _, code in assets['CRAFT.css'])
assert '#craft-desk-picker' in css and '#craft-insert-preview' in css

## Live `fastcdp` Verification

In [ ]:
#| live
import json
from fastcdp import Rungs
from craft_dev.cdp import open_solveit

async with open_solveit() as (page, _, page_errors):
    rungs = Rungs(page)

    async with rungs('insert preview position and hotkey'):
        assert await page.eval(r"""(() => {
          const t = document.getElementById('craft-insert-preview'), sc = document.getElementById('dialog-container');
          return !!(t && sc && t.getBoundingClientRect().right <= sc.getBoundingClientRect().right);
        })()""")
        prev_type = await page.eval("window.craftInsertPreview.type()")
        await page.press('K', mod=True, shift=True)
        await page.wait_for("window.craftInsertPreview.type() === 'Note'", timeout=3)
        await page.eval(f"window.craftInsertPreview.choose({json.dumps(prev_type)})")

    assert not page_errors, f"Uncaught page errors: {page_errors}"